In [1]:
#---------------------------------------------------------
# 1. LIBRARIES
#---------------------------------------------------------

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from hdbscan import HDBSCAN
from sklearn.model_selection import ParameterGrid

In [2]:
#---------------------------------------------------------
# 2. DATA PREPROCESSING
#---------------------------------------------------------

import sys
import os

main_dir = os.path.abspath('..')

sys.path.append(main_dir + '/utils')

from preprocessing import preprocess

train_data = pd.read_csv('../data/adult_train_cleaned.csv')

x_train = train_data.drop(['income', 'race', 'sex'], axis = 1)
y_train = train_data['income']

preprocessor = preprocess(x_train)

In [6]:

#---------------------------------------------------------
# 3. HYPERPARAMETER TUNING & CLUSTERING EVALUATION
#---------------------------------------------------------

x_train_preprocessed = preprocessor.fit_transform(x_train)

from cluster_eval import evaluate_clustering

#---------------------------------------------------------
# A. K-MEANS HYPERPARAMETER TUNING
#---------------------------------------------------------

kmeans_param_grid = {
    'algorithm': ['lloyd', 'elkan'],
    'n_clusters': range(2, 11),
    'init': ['k-means++', 'random'],
    'n_init': [10, 20, 30],
    'max_iter': [100, 300, 500],
    'tol': [1e-4, 1e-5, 1e-6]
}

kmeans_results = []

for params in ParameterGrid(kmeans_param_grid):

    model = KMeans(
        **params,
        random_state = 42
    )

    labels = model.fit_predict(x_train_preprocessed)

    scores = evaluate_clustering(
        x_train_preprocessed,
        labels,
        y_true = y_train
    )

    kmeans_results.append({
        **params,
        **scores,
        'Inertia': model.inertia_
    })

kmeans_results = pd.DataFrame(kmeans_results)

kmeans_results = kmeans_results.sort_values(
    by = 'Silhouette',
    ascending = False
)

print(kmeans_results.head())

#---------------------------------------------------------
# B. DBSCAN HYPERPARAMETER TUNING
#---------------------------------------------------------

dbscan_param_grid = {
    'eps': [0.1, 0.2, 0.3, 0.5, 0.7, 1.0],
    'min_samples': [3, 5, 10, 15, 20],
    'metric': ['euclidean', 'manhattan'],
    'leaf_size': [20, 30, 40, 50]
}

dbscan_results = []

for params in ParameterGrid(dbscan_param_grid):

    model = DBSCAN(
        **params,
        algorithm = 'auto',
        n_jobs = -1
    )

    labels = model.fit_predict(x_train_preprocessed)

    scores = evaluate_clustering(
        x_train_preprocessed,
        labels,
        y_true = y_train
    )

    dbscan_results.append({
        **params,
        **scores
    })

dbscan_results = pd.DataFrame(dbscan_results)

dbscan_results = dbscan_results.sort_values(
    by = 'Silhouette',
    ascending = False
)

print(dbscan_results.head())

#---------------------------------------------------------
# C. COMPARISON OF THE TWO ALGORITHMS
#---------------------------------------------------------

metric_cols = [
    'Clusters',
    'Noise (%)',
    'Silhouette',
    'Davies-Bouldin',
    'Calinski-Harabasz'
]

comparison = pd.DataFrame({
    'K-Means': kmeans_results.iloc[0][metric_cols],
    'DBSCAN': dbscan_results.iloc[0][metric_cols]
})

print(comparison)

#---------------------------------------------------------
# D. EXPORTING RESULTS
#---------------------------------------------------------

kmeans_results.to_csv('../data/kmeans_results.csv', index = False)
dbscan_results.to_csv('../data/dbscan_results.csv', index = False)
comparison.to_csv('../data/clustering_comparison.csv')

    algorithm    init  max_iter  n_clusters  n_init       tol  Clusters  \
245     lloyd  random       100           2      10  0.000001         2   
893     elkan  random       500           2      10  0.000001         2   
326     lloyd  random       300           2      10  0.000001         2   
892     elkan  random       500           2      10  0.000010         2   
891     elkan  random       500           2      10  0.000100         2   

     Noise (%)  Silhouette  Davies-Bouldin  Calinski-Harabasz  \
245        0.0    0.403491        0.993145        4952.182618   
893        0.0    0.403491        0.993145        4952.182618   
326        0.0    0.403491        0.993145        4952.182618   
892        0.0    0.403491        0.993145        4952.182618   
891        0.0    0.403491        0.993145        4952.182618   

     Adjusted Rand Score        Inertia  
245             0.063535  216683.088393  
893             0.063535  216683.088393  
326             0.063535  216683

In [7]:
#---------------------------------------------------------
# 4. MODEL TRAINING
#---------------------------------------------------------

kmeans = KMeans(
    n_clusters = 2,
    init = 'random',
    n_init = 10,
    max_iter = 300,
    tol = 1e-6,
    algorithm = 'lloyd',
    random_state = 42
)

cohort_labels = kmeans.fit_predict(x_train_preprocessed)

audit_data = train_data[['race', 'sex', 'income']].copy()
audit_data['cohort'] = cohort_labels

In [8]:
#---------------------------------------------------------
# 5. MODEL PERSISTENCE
#---------------------------------------------------------

joblib.dump(kmeans, '../models/cohort_kmeans.joblib')

['../models/cohort_kmeans.joblib']